In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [1]:
# !pip install pymongo

In [2]:
import pandas as pd
import json
from pathlib import Path
from pymongo import MongoClient

In [3]:
# Connect to local MongoDB
client = MongoClient("mongodb://localhost:27017/")

# Create / connect to database
mongodb = client["uk_entity_review"]

# Create / connect to collection
policy_collection = mongodb["policy_documents"]
source_registry_collection = mongodb["source_registry"]

print("Connected to MongoDB")
print("Database:", mongodb.name)
print("Collection:", policy_collection.name)
print("Collection:", source_registry_collection.name)

Connected to MongoDB
Database: uk_entity_review
Collection: policy_documents
Collection: source_registry


In [4]:
PROJECT_ROOT = Path.cwd()
DOCS_CURATED_DIR = PROJECT_ROOT / "data" / "docs_curated"
DOCS_RAW_DIR = PROJECT_ROOT / "data" / "docs_raw"

document_files = [
    "entity_review_policy.md",
    "risk_flag_guidelines.md",
    "manual_review_checklist.md",
]

documents = []

for i, filename in enumerate(document_files, start=1):
    file_path = DOCS_CURATED_DIR / filename
    content = file_path.read_text(encoding="utf-8")

    document = {
        "doc_id": f"policy_{i:03d}",
        "title": filename.replace(".md", "").replace("_", " ").title(),
        "doc_type": "policy_guidance",
        "source_type": "curated_markdown",
        "source_file": filename,
        "content": content,
    }
    documents.append(document)

source_registry_path = DOCS_RAW_DIR / "source_registry.json"
source_registry_documents = json.loads(source_registry_path.read_text(encoding="utf-8"))

print(f"Loaded {len(documents)} policy documents")
print(f"Loaded {len(source_registry_documents)} source registry records")

Loaded 3 policy documents
Loaded 4 source registry records


In [5]:
policy_collection.delete_many({})
policy_insert_result = policy_collection.insert_many(documents)

source_registry_collection.delete_many({})
source_registry_insert_result = source_registry_collection.insert_many(source_registry_documents)

print("Inserted policy document IDs:")
print(policy_insert_result.inserted_ids)

print("Inserted source registry IDs:")
print(source_registry_insert_result.inserted_ids)

Inserted policy document IDs:
[ObjectId('69bd7857fe930c18383d1810'), ObjectId('69bd7857fe930c18383d1811'), ObjectId('69bd7857fe930c18383d1812')]
Inserted source registry IDs:
[ObjectId('69bd7857fe930c18383d1813'), ObjectId('69bd7857fe930c18383d1814'), ObjectId('69bd7857fe930c18383d1815'), ObjectId('69bd7857fe930c18383d1816')]


In [6]:
documents_preview = list(
    policy_collection.find(
        {},
        {"_id": 0, "doc_id": 1, "title": 1, "doc_type": 1, "source_file": 1}
    )
)

pd.DataFrame(documents_preview)


,doc_id,title,doc_type,source_file
0,policy_001,Entity Review Policy,policy_guidance,entity_review_policy.md
1,policy_002,Risk Flag Guidelines,policy_guidance,risk_flag_guidelines.md
2,policy_003,Manual Review Checklist,policy_guidance,manual_review_checklist.md


In [7]:
search_results = list(
    policy_collection.find(
        {"content": {"$regex": "outstanding", "$options": "i"}},
        {"_id": 0, "doc_id": 1, "title": 1}
    )
)

print(search_results)

[{'doc_id': 'policy_002', 'title': 'Risk Flag Guidelines'}, {'doc_id': 'policy_003', 'title': 'Manual Review Checklist'}]


In [8]:
source_registry_preview = list(
    source_registry_collection.find(
        {},
        {"_id": 0, "source_id": 1, "title": 1, "source_type": 1, "category": 1}
    )
)

pd.DataFrame(source_registry_preview)

,source_id,title,source_type,category
0,src_001,"Know Your Customer Guidance, Accessible Version",gov_uk_webpage,customer_guidance
1,src_002,UK Financial Sanctions General Guidance,gov_uk_webpage,financial_sanctions_guidance
2,src_003,Assessing and Reducing the Risk of Money Laund...,pdf_document,money_laundering_risk_guidance
3,src_004,Corporate Transparency and Register Reform Whi...,pdf_document,corporate_transparency
